In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import pandas as pd

from pandas import ExcelWriter

import pdfplumber

from time import sleep

import datetime

from selenium import webdriver

from selenium.webdriver.common.by import By

import os

import re


In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'HK HKMA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.7")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)


Running HK HKMA Web Scraping Tool v.1.7


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


regdict={
    regulatorName + ' 1': 'https://www.hkma.gov.hk/eng/key-functions/banking/banking-regulatory-and-supervisory-regime/the-three-tier-banking-system/', 

	regulatorName + ' 2': 'https://www.hkma.gov.hk/eng/key-functions/banking/banking-regulatory-and-supervisory-regime/the-three-tier-banking-system/', 

	regulatorName + ' 3': 'https://www.hkma.gov.hk/eng/key-functions/banking/banking-regulatory-and-supervisory-regime/the-three-tier-banking-system/', 

	regulatorName + ' 4': 'https://www.hkma.gov.hk/eng/key-functions/banking/banking-regulatory-and-supervisory-regime/the-three-tier-banking-system/', 

	regulatorName + ' 5': 'https://www.hkma.gov.hk/eng/regulatory-resources/list-of-ai-related-trustees/', 
   
	regulatorName + ' 6': 'https://www.hkma.gov.hk/eng/regulatory-resources/registers/register-of-svf-licensees/'
   }



Typology = {

            "HK HKMA 1": "List of licensed banks",

            "HK HKMA 2": "List of restricted licence banks", 

		    "HK HKMA 3": "List of deposit-taking companies", 

            "HK HKMA 4": "List of local representative offices",

		    "HK HKMA 5": "List of AI-related Trustees", 

            "HK HKMA 6": "Register of SVF Licensees",
            }



sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}



location = ['Incorporated in Hong Kong', 'Incorporated outside Hong Kong', 'Offices in Hong Kong']

processdate = now.strftime('%Y-%m-%d')




In [5]:


# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]


# %%


In [ ]:

# %%
#------------------------------------------------ Begin_Main ----------------------------------------
df_total = pd.DataFrame(sqldict)
for k, reg in enumerate(regdict):
    print(f'Working {k+1}/{len(regdict)} | {reg} | {Typology[reg]}')
    driver.get(regdict[reg])
    sleep(10)
    soup = BeautifulSoup(driver.page_source, 'html.parser')  
    sleep(5)
    if reg == 'HK HKMA 1' or reg=='HK HKMA 2'or reg=='HK HKMA 3'or reg=='HK HKMA 4':
        driver.find_element(By.XPATH, f"//a[contains(text(),'{Typology[reg]}')]").click()
        sleep(3)
        file =  check_dowload_files(tempfolder, "xls")
        filePath = os.path.join(tempfolder, file)
        df = pd.read_excel(filePath)
    if reg == 'HK HKMA 1':
        df1 = pd.DataFrame(sqldict)
        valid_df = df.iloc[:,0]
        location = ['Incorporated in Hong Kong', 'List of Digital Banks','Incorporated outside Hong Kong']
        banks = []
        category = []
        for i in range(len(location)):
            start_index = valid_df[valid_df.str.contains(location[i], na=False)].index[0].item()
            print(f"[INFO] -- Deal with {location[i]} data -- ")
            print(f"Data starts at index: {start_index}")
            end_index = valid_df[start_index+1:].isna().idxmax()
            print(f"Data ends at index: {end_index}")
            infos_list = valid_df[start_index+1:end_index]
            infos_list = infos_list[~infos_list.str.strip().str.contains('ALSO KNOWN AS', na=False)]
            banks.extend([info.strip() for info in infos_list.tolist()])
            category.extend([valid_df[start_index]] * len(infos_list))
        # Combine banks and category into a DataFrame
        category = [x.replace('*', '').replace(':', '').lstrip() for x in category]
        df_combined = pd.DataFrame({'Bank': banks, 'Category': category})
        df_combined['Bank'] = df_combined['Bank'].apply(lambda x: re.sub(r'^\d+\.\s*', '', x))
        df1['Name'] = df_combined['Bank']
        df1['Typology'] = df_combined['Category']
        df1['ListProcessDate'] = processdate
        df1['ListName'] = Typology[reg]
        df1['RegCtry'] = reg.split(' ')[0]
        df1['RegCode'] = reg.split(' ')[1]
        df1['ListCode'] = reg.split(' ')[-1]
        df1['RegulationType'] = 'Regulated'
        df1.fillna('')
        df_total = pd.concat([df_total, df1], axis=0)
    elif reg == 'HK HKMA 2' or reg == 'HK HKMA 3':
        df2 = pd.DataFrame(sqldict)
        valid_df = df.iloc[:,0]
        location = ['Incorporated in Hong Kong','Incorporated outside Hong Kong']
        banks = []
        category = []
        for i in range(len(location)):
            start_index = valid_df[valid_df.str.contains(location[i], na=False)].index[0]
            print(f"[INFO] -- Deal with {location[i]} data -- ")
            print(f"Data starts at index: {start_index}")
            end_index = valid_df[start_index+1:].isna().idxmax()
            print(f"Data ends at index: {end_index}")
            infos_list = valid_df[start_index+1:end_index]
            infos_list = infos_list[~infos_list.str.strip().str.contains('ALSO KNOWN AS', na=False)]
            banks.extend([info.strip() for info in infos_list.tolist()])
            category.extend([valid_df[start_index]] * len(infos_list))
        # Combine banks and category into a DataFrame
        category = [x.replace('*', '').replace(':', '').lstrip() for x in category]
        df_combined = pd.DataFrame({'Bank': banks, 'Category': category})

        df2['Name'] = df_combined['Bank']
        df2['Typology'] = df_combined['Category']
        df2['ListProcessDate'] = processdate
        df2['ListName'] = Typology[reg]
        df2['RegCtry'] = reg.split(' ')[0]
        df2['RegCode'] = reg.split(' ')[1]
        df2['ListCode'] = reg.split(' ')[-1]
        df2['RegulationType'] = 'Regulated'
        df2.fillna('')
        df_total = pd.concat([df_total, df2], axis=0)        
    elif reg == 'HK HKMA 4':
        df4 = pd.DataFrame(sqldict)
        banks = []
        first_nan_index = df.isna().idxmax()
        next_nan_index = df[first_nan_index[0]+1:].isna().idxmax()
        infos_list = df[first_nan_index[0]+1:next_nan_index[0]]
        infos_list = infos_list.iloc[:,0]
        infos_list = infos_list[~infos_list.str.strip().str.contains('ALSO KNOWN AS', na=False)]
        banks.extend([info.strip() for info in infos_list.tolist()])
        df_combined = pd.DataFrame({'Bank': banks})

        df4['Name'] = df_combined['Bank']
        df4['ListProcessDate'] = processdate
        df4['ListName'] = Typology[reg]
        df4['RegCtry'] = reg.split(' ')[0]
        df4['RegCode'] = reg.split(' ')[1]
        df4['ListCode'] = reg.split(' ')[-1]
        df4['RegulationType'] = 'Regulated'
        df4.fillna('')
        df_total = pd.concat([df_total, df4], axis=0)  
    elif reg == 'HK HKMA 5':
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        tables = soup.find_all('table')
        for table in tables:
            tbody = table.find('tbody')
            trs = tbody.find_all('tr')
            #print(len(trs))
            for tr in trs:
                tds = tr.find_all('td')
                #print(len(tds))
                if len(tds)>0:
                    if len(tds)>3:
                        name = tds[0].text
                        address = tds[1].text
                        website = tds[2].text
                        morther_company_name = tds[3].text
                        # print(name)
                        # print(address)
                        website = website if len(website)>3 else ''
                        # print(website)
                        # print(morther_company_name)
                    else:
                        name = tds[0].text
                        address = tds[1].text
                        website = tds[2].text
                        morther_company_name=''
                        # print(name)
                        # print(address)
                        # print(website)
                    sqldict['Name'].append(name)
                    sqldict['Address_1'].append(address)
                    sqldict['Website'].append(website)
                    sqldict['Name - Mother Company'].append(morther_company_name)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
                df5=pd.DataFrame(sqldict)
                df5 = df5.drop_duplicates()
                df_total = pd.concat([df_total, df5], axis=0) 
    elif reg == 'HK HKMA 6':
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

         'Phone - Mother company': [], 'Check': []}

        # Table in PDF files
        element = soup.find('div', {'data-id': '7de193b'})
        document_link = element.find('a')['href']
        driver.get('https://www.hkma.gov.hk/'+document_link)
        sleep(10)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        pages_text = list()
        bold_lines = list()
        name_list = list()
        address_list = list()
        svf_list = list()
        date_list = list()
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        df6_2 = pd.DataFrame(sqldict)

        with pdfplumber.open(dl_files[0]) as pdf:
            for page in pdf.pages:
                bold_text = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" in obj["fontname"]).extract_text()
                page_content = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"]).extract_text()
                # Set Adress column 'A' is X0
                name = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and obj['x0'] <= 305).extract_text()
                address = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and 305<obj['x0'] <= 720).extract_text()
                svf_id = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and 721<obj['x0'] <= 751).extract_text()
                effective_date = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and 751.44096 <=obj['x0'] <= 783).extract_text()
                if bold_text is not None:
                    bold_lines.extend([ele.strip() for ele in bold_text.split('\n') if len(ele) > 0])            
                #pages_text.append(page.extract_text().strip())
                if page_content is not None:
                    pages_text.extend([ele.strip() for ele in page_content.split('\n') if len(ele) > 0])  
                if name is not None:
                    name_list.extend([ele.strip() for ele in name.split('\n') if len(ele) > 0])
                if address is not None:
                    address_list.extend([ele.strip() for ele in address.split('\n') if len(ele) > 0])
                if svf_id is not None:
                    svf_list.extend([ele.strip() for ele in svf_id.split('\n') if len(ele) > 0])
                if effective_date is not None:
                    date_list.extend([ele.strip() for ele in effective_date.split('\n') if len(ele) > 0])
                    
        name_filtered_list = [item for item in name_list if 'Updated:' not in item]
        date_filtered_list = [item for item in date_list if 'Page' not in item]

        df6_2['Name'] = name_filtered_list
        df6_2['RegulationDate'] = date_filtered_list
        df6_2['Address_1'] = address_list
        df6_2['InternalID_1'] = svf_list
        df6_2['InternalID_1_type'] = ('Licence Number')
        df6_2['ListProcessDate'] = processdate
        df6_2['ListName'] = Typology[reg]
        df6_2['RegCtry'] = reg.split(' ')[0]
        df6_2['RegCode'] = reg.split(' ')[1]
        df6_2['ListCode'] = reg.split(' ')[-1]
        df6_2['RegulationType'] = 'Regulated'
        df_total = pd.concat([df_total, df6_2], axis=0) 
        
        # Data in Website table
        tables = soup.find_all('table')
        for table in tables:
            tbody = table.find('tbody')
            trs = tbody.find_all('tr')
            #print(len(trs))
            for tr in trs:
                tds = tr.find_all('td')
                #print(len(tds))
                if len(tds)>0:
                    name_web = tds[0].find('strong').text
                    address_web = tds[0].find_all('p')[-1].text.replace('Address:\n','')
                    svf_id_web = tds[1].text.lstrip().rstrip()
                    effective_date_web = tds[2].text.lstrip().rstrip()

                    # print(name)
                    # print(address)
                    # print(svf_id)
                    # print(effective_date)
                    sqldict['Name'].append(name_web)
                    sqldict['Address_1'].append(address_web)
                    sqldict['InternalID_1'].append(svf_id_web)
                    sqldict['RegulationDate'].append(effective_date_web)
                    sqldict['InternalID_1_type'].append('Licence Number')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['RegulationType'].append('Regulated')
                sqldict = bourange_same_length_array(sqldict)
                df6=pd.DataFrame(sqldict)
                df6 = df6.drop_duplicates()
                df_total = pd.concat([df_total, df6], axis=0)
                df_total = df_total.fillna('') 
                

    sleep(5)
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))

Working 1/6 | HK HKMA 1 | List of licensed banks
[INFO] : - xls file = ['list_of_lb.xls'])
[INFO] -- Deal with Incorporated in Hong Kong data -- 
Data starts at index: 2
Data ends at index: 35
[INFO] -- Deal with List of Digital Banks data -- 
Data starts at index: 37
Data ends at index: 46
[INFO] -- Deal with Incorporated outside Hong Kong data -- 
Data starts at index: 62
Data ends at index: 198
Working 2/6 | HK HKMA 2 | List of restricted licence banks
[INFO] : - xls file = ['list_of_rlb.xls'])
[INFO] -- Deal with Incorporated in Hong Kong data -- 
Data starts at index: 2
Data ends at index: 12
[INFO] -- Deal with Incorporated outside Hong Kong data -- 
Data starts at index: 28
Data ends at index: 35
Working 3/6 | HK HKMA 3 | List of deposit-taking companies
[INFO] : - xls file = ['list_of_dtc.xls'])
[INFO] -- Deal with Incorporated in Hong Kong data -- 
Data starts at index: 2
Data ends at index: 14
[INFO] -- Deal with Incorporated outside Hong Kong data -- 
Data starts at index: 3

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_21180\1475428810.py:81: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  next_nan_index = df[first_nan_index[0]+1:].isna().idxmax()
C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_21180\1475428810.py:82: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  infos_list = df[first_nan_index[0]+1:next_nan_index[0]]


Working 5/6 | HK HKMA 5 | List of AI-related Trustees
Working 6/6 | HK HKMA 6 | Register of SVF Licensees


In [ ]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

# df=pd.DataFrame(sqldict)
df_total = df_total.drop_duplicates()

df_total.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
    

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_21180\1514217216.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df_total.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [11]:
df_total.to_csv('total_1_6_ver2.csv')

In [10]:
df_total = df_total.drop_duplicates()

In [ ]:
pages_text = list()
bold_lines = list()
name_list = list()
address_list = list()
svf_list = list()
date_list = list()
dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
df6_2 = pd.DataFrame(sqldict)

with pdfplumber.open(dl_files[0]) as pdf:
    for page in pdf.pages:
        bold_text = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" in obj["fontname"]).extract_text()
        page_content = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"]).extract_text()
        # Set Adress column 'A' is X0
        name = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and obj['x0'] <= 305).extract_text()
        address = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and 305<obj['x0'] <= 720).extract_text()
        svf_id = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and 721<obj['x0'] <= 751).extract_text()
        effective_date = page.filter(lambda obj: obj["object_type"] == "char" and "Bold" not in obj["fontname"] and 751.44096 <=obj['x0'] <= 783).extract_text()
        if bold_text is not None:
            bold_lines.extend([ele.strip() for ele in bold_text.split('\n') if len(ele) > 0])            
        #pages_text.append(page.extract_text().strip())
        if page_content is not None:
            pages_text.extend([ele.strip() for ele in page_content.split('\n') if len(ele) > 0])
            
        if name is not None:
            name_list.extend([ele.strip() for ele in name.split('\n') if len(ele) > 0])
        if address is not None:
            address_list.extend([ele.strip() for ele in address.split('\n') if len(ele) > 0])
        if svf_id is not None:
            svf_list.extend([ele.strip() for ele in svf_id.split('\n') if len(ele) > 0])
        if effective_date is not None:
            date_list.extend([ele.strip() for ele in effective_date.split('\n') if len(ele) > 0])
            
name_filtered_list = [item for item in name_list if 'Updated:' not in item]
date_filtered_list = [item for item in date_list if 'Page' not in item]

df6_2['Name'] = name_filtered_list
df6_2['RegulationDate'] = date_filtered_list
df6_2['Address_1'] = address_list
df6_2['InternalID_1'] = svf_list
df6_2['InternalID_1_type'] = ('Licence Number')
df6_2['ListProcessDate'] = processdate
df6_2['ListName'] = Typology[reg]
df6_2['RegCtry'] = reg.split(' ')[0]
df6_2['RegCode'] = reg.split(' ')[1]
df6_2['ListCode'] = reg.split(' ')[-1]
df6_2['RegulationType'] = 'Regulated'
df_total = pd.concat([df_total, df6_2], axis=0) 

In [ ]:

import pdfplumber
min_width = 52.92
max_width = 306.6
min_height = 523.99008
max_height = 10000
with pdfplumber.open(dl_files[0]) as pdf:
    # Select the page you want to work with
    page = pdf.pages[0]
    
    # Extract the characters from the page
    chars = page.chars
    
    # Filter the characters based on the criteria
    Bold_chars = [char for char in chars if char["object_type"] == "char" and 'BoldMT' in char['fontname']]
    Text_chars = [char for char in chars if char["object_type"] == "char" and 'BoldMT' not in char['fontname']]
    filtered_chars = [char for char in chars if min_width <= char['x0'] <= max_width and 'BoldMT' not in char['fontname']]
    # Extract text lines from the filtered characters
    text_lines = [char['text'] for char in filtered_chars]
    # Print the extracted text lines
    for i in range(len(Bold_chars)):
        print(i,Bold_chars[i]['text'])
        print(i,Bold_chars[i]['x0'])
    
    # print(Bold_chars[104]['text'])
    # print(Bold_chars[104]['x0'])
    # print(Bold_chars[104]['y0'])
    # print(Bold_chars[104]['y1'])
    #print(text_lines)

